In [12]:
import requests
import pandas as pd
import numpy as np
import time

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

def fetch_json(url, max_retries=3):
    for _ in range(max_retries):
        try:
            res = requests.get(url, headers=headers, timeout=12)
            if res.status_code == 200:
                return res.json()
        except Exception:
            pass
        time.sleep(1.5)
    return None

# 1. Fetch bootstrap static metadata
data = fetch_json("https://fantasy.premierleague.com/api/bootstrap-static/")
if not data:
    raise RuntimeError("Failed to fetch static FPL API data.")

raw_players = pd.DataFrame(data['elements'])
teams = pd.DataFrame(data['teams'])[['id', 'name']].rename(columns={'id': 'team', 'name': 'team_name'})
positions = pd.DataFrame(data['element_types'])[['id', 'singular_name_short']].rename(
    columns={'id': 'element_type', 'singular_name_short': 'pos_name'}
)

players_df = raw_players.merge(teams, on='team').merge(positions, on='element_type')

# 2. Map Fixture Difficulty (FDR) and Home/Away status
fixtures = fetch_json("https://fantasy.premierleague.com/api/fixtures/")
fdr_map = {}
if fixtures:
    next_fixtures = [f for f in fixtures if not f.get('finished')]
    if next_fixtures:
        next_gw = next_fixtures[0].get('event')
        for fix in [f for f in next_fixtures if f.get('event') == next_gw]:
            fdr_map[fix['team_h']] = {'opponent': fix['team_a'], 'fdr': fix['team_h_difficulty'], 'was_home': 1}
            fdr_map[fix['team_a']] = {'opponent': fix['team_h'], 'fdr': fix['team_a_difficulty'], 'was_home': 0}

# 3. Pull historical logs for active starters
active_players = players_df[players_df['minutes'] > 0].copy()
all_gw_data = []

print(f"Ingesting match histories for {len(active_players)} players...")
for idx, (_, player) in enumerate(active_players.iterrows()):
    p_id, p_team = player['id'], player['team']
    summary = fetch_json(f"https://fantasy.premierleague.com/api/element-summary/{p_id}/")
    
    p_fdr = fdr_map.get(p_team, {}).get('fdr', 3)
    p_home = fdr_map.get(p_team, {}).get('was_home', 1)
    
    if summary and 'history' in summary:
        for match in summary['history']:
            match.update({
                'player_id': p_id,
                'web_name': player['web_name'],
                'team_name': player['team_name'],
                'element_type': player['element_type'],
                'pos_name': player['pos_name'],
                'now_cost': player['now_cost'],
                'chance_next_round': player.get('chance_of_playing_next_round', 100) or 100,
                'fdr': p_fdr,
                'was_home': p_home
            })
            all_gw_data.append(match)
    time.sleep(0.01)

gw_df = pd.DataFrame(all_gw_data)

# 4. Exponentially Weighted Moving Averages (EWMA) Feature Extraction
gw_df = gw_df.sort_values(by=['player_id', 'round'])
metrics = ['total_points', 'minutes', 'expected_goals', 'expected_assists', 'bps', 'ict_index']

for m in metrics:
    if m in gw_df.columns:
        gw_df[m] = pd.to_numeric(gw_df[m], errors='coerce').fillna(0)
        gw_df[f'ewma_{m}'] = gw_df.groupby('player_id')[m].transform(
            lambda x: x.shift(1).ewm(alpha=0.4, min_periods=1).mean()
        )

gw_df['target_next_points'] = gw_df.groupby('player_id')['total_points'].shift(-1)
print(f"Pipeline ready. Data shape: {gw_df.shape}")

Ingesting match histories for 339 players...
Pipeline ready. Data shape: (674, 56)


In [13]:
from xgboost import XGBRegressor
import numpy as np

features = [f'ewma_{m}' for m in metrics] + ['now_cost', 'fdr', 'was_home']
for col in features:
    gw_df[col] = pd.to_numeric(gw_df[col], errors='coerce').fillna(0)

position_models = {}
train_df = gw_df.dropna(subset=['target_next_points']).copy()

# Train dedicated XGBoost models per position (GK=1, DEF=2, MID=3, FWD=4)
for pos_id, pos_code in [(1, 'GK'), (2, 'DEF'), (3, 'MID'), (4, 'FWD')]:
    sub_train = train_df[train_df['element_type'] == pos_id]
    
    if len(sub_train) > 10:
        X = sub_train[features]
        y = sub_train['target_next_points']
    else:
        X = gw_df[gw_df['element_type'] == pos_id][features]
        y = gw_df[gw_df['element_type'] == pos_id]['total_points']
    
    reg = XGBRegressor(n_estimators=120, learning_rate=0.03, max_depth=4, random_state=42)
    reg.fit(X, y)
    position_models[pos_id] = reg

print("Position-specific XGBoost Regressors successfully trained!")

Position-specific XGBoost Regressors successfully trained!


In [14]:
# Extract current state per player
latest_df = gw_df.groupby('player_id').last().reset_index()

# Run inference through positional models
preds = []
for _, row in latest_df.iterrows():
    p_pos = row['element_type']
    model = position_models.get(p_pos)
    
    feat_vector = pd.DataFrame([row[features]])
    raw_pred = model.predict(feat_vector)[0]
    
    # Calculate Expected Minutes (xM) weight factor
    availability_factor = (row['chance_next_round'] / 100.0)
    minutes_factor = min(row['ewma_minutes'] / 90.0, 1.0) if row['ewma_minutes'] > 0 else 0.5
    
    # Adjust prediction by availability risk multiplier
    adjusted_pred = raw_pred * availability_factor * (0.3 + 0.7 * minutes_factor)
    preds.append(max(0.0, adjusted_pred))

latest_df['predicted_next_points'] = preds

# Export detailed predictions to CSV
export_cols = ['web_name', 'pos_name', 'team_name', 'now_cost', 'predicted_next_points']
latest_df[export_cols].sort_values(by='predicted_next_points', ascending=False).to_csv('next_gw_predictions.csv', index=False)
print("Updated `next_gw_predictions.csv` with position-weighted projections.")

Updated `next_gw_predictions.csv` with position-weighted projections.


In [19]:
import pandas as pd

# 1. Get the most recent match state for every active player
latest_states = gw_df.groupby('player_id').last().reset_index()

# Normalize position names (FPL API uses 'GKP' -> convert to 'GK')
latest_states['pos_name'] = latest_states['pos_name'].replace({'GKP': 'GK'})

# 2. Predict next Gameweek points using position-specific XGBoost models
preds = []
for _, row in latest_states.iterrows():
    p_pos = row['element_type']
    model = position_models.get(p_pos)
    
    feat_vector = pd.DataFrame([row[features]])
    raw_pred = model.predict(feat_vector)[0] if model else 0.0
    
    # Apply Availability & Minutes risk multiplier
    availability_factor = (row['chance_next_round'] / 100.0)
    minutes_factor = min(row['ewma_minutes'] / 90.0, 1.0) if row['ewma_minutes'] > 0 else 0.5
    
    adjusted_pred = raw_pred * availability_factor * (0.3 + 0.7 * minutes_factor)
    preds.append(max(0.0, adjusted_pred))

latest_states['predicted_next_points'] = preds

# Export normalized predictions for the PuLP optimization step
export_cols = ['web_name', 'pos_name', 'team_name', 'now_cost', 'predicted_next_points']
predictions_df = latest_states[export_cols].sort_values(by='predicted_next_points', ascending=False)
predictions_df.to_csv('next_gw_predictions.csv', index=False)

print("Updated `next_gw_predictions.csv` with normalized position tags!")
display(predictions_df.head(10))

Updated `next_gw_predictions.csv` with normalized position tags!


,web_name,pos_name,team_name,now_cost,predicted_next_points
258,Gibbs-White,MID,Nott'm Forest,79,11.458347
157,Ajayi,DEF,Hull City,40,2.015765
324,Mendy,DEF,Hull City,40,2.015765
208,Matheus N.,DEF,Man City,60,1.710155
250,Osula,FWD,Newcastle,60,1.612466
158,Coyle,DEF,Hull City,40,1.601130
236,Schär,DEF,Newcastle,50,1.523597
259,Anderson,MID,Man City,64,1.363916
47,Rodríguez,FWD,Bournemouth,60,1.238477
192,Jacquet,DEF,Liverpool,50,1.232230


In [20]:
import joblib
import pandas as pd

# 1. Save model weights to disk
joblib.dump(model, 'fpl_xgboost_model.pkl')
print("Model saved to disk as 'fpl_xgboost_model.pkl'")

# 2. Safely filter available columns for CSV export
desired_cols = ['web_name', 'pos_name', 'team_name', 'now_cost', 'predicted_next_points']
available_cols = [col for col in desired_cols if col in latest_states.columns]

# Export available player predictions to CSV
latest_states[available_cols] \
    .sort_values(by='predicted_next_points', ascending=False) \
    .to_csv('next_gw_predictions.csv', index=False)

print(f"Predictions exported to 'next_gw_predictions.csv' with columns: {available_cols}")

Model saved to disk as 'fpl_xgboost_model.pkl'
Predictions exported to 'next_gw_predictions.csv' with columns: ['web_name', 'pos_name', 'team_name', 'now_cost', 'predicted_next_points']


In [22]:
import pandas as pd
from pulp import LpMaximize, LpProblem, LpVariable, lpSum, LpStatus

# 1. Load predictions
df = pd.read_csv("next_gw_predictions.csv")
df['pos_name'] = df['pos_name'].replace({'GKP': 'GK'})

model = LpProblem(name="fpl-squad-optimizer", sense=LpMaximize)
indices = df.index.tolist()

# 2. Decision Variables
x_start = {i: LpVariable(name=f"start_{i}", cat="Binary") for i in indices}
x_bench = {i: LpVariable(name=f"bench_{i}", cat="Binary") for i in indices}
x_capt  = {i: LpVariable(name=f"capt_{i}",  cat="Binary") for i in indices}

# 3. Objective Function: Maximize Starting XI + Captain Boost
model += lpSum(
    (df.loc[i, "predicted_next_points"] * x_start[i]) + 
    (df.loc[i, "predicted_next_points"] * x_capt[i]) 
    for i in indices
)

# 4. Strict Constraints
# Total Budget <= £100.0m (1000 in API units)
model += lpSum(df.loc[i, "now_cost"] * (x_start[i] + x_bench[i]) for i in indices) <= 1000, "Budget_Cap"

# Squad Sizes: 11 Starters + 4 Bench = 15 Total Players
model += lpSum(x_start[i] for i in indices) == 11, "Must_Have_11_Starters"
model += lpSum(x_bench[i] for i in indices) == 4, "Must_Have_4_Bench"
model += lpSum(x_capt[i] for i in indices) == 1, "Single_Captain"

for i in indices:
    model += x_start[i] + x_bench[i] <= 1, f"One_Slot_Per_Player_{i}"
    model += x_capt[i] <= x_start[i], f"Captain_Must_Start_{i}"

# Positional Squad Breakdown (2 GK, 5 DEF, 5 MID, 3 FWD)
model += lpSum(x_start[i] + x_bench[i] for i in indices if df.loc[i, "pos_name"] == "GK") == 2, "Squad_GK"
model += lpSum(x_start[i] + x_bench[i] for i in indices if df.loc[i, "pos_name"] == "DEF") == 5, "Squad_DEF"
model += lpSum(x_start[i] + x_bench[i] for i in indices if df.loc[i, "pos_name"] == "MID") == 5, "Squad_MID"
model += lpSum(x_start[i] + x_bench[i] for i in indices if df.loc[i, "pos_name"] == "FWD") == 3, "Squad_FWD"

# Starting XI Formation (1 GK, 3-5 DEF, 3-5 MID, 1-3 FWD)
model += lpSum(x_start[i] for i in indices if df.loc[i, "pos_name"] == "GK") == 1, "Starting_GK"
model += lpSum(x_start[i] for i in indices if df.loc[i, "pos_name"] == "DEF") >= 3, "Min_DEF"
model += lpSum(x_start[i] for i in indices if df.loc[i, "pos_name"] == "DEF") <= 5, "Max_DEF"
model += lpSum(x_start[i] for i in indices if df.loc[i, "pos_name"] == "MID") >= 3, "Min_MID"
model += lpSum(x_start[i] for i in indices if df.loc[i, "pos_name"] == "MID") <= 5, "Max_MID"
model += lpSum(x_start[i] for i in indices if df.loc[i, "pos_name"] == "FWD") >= 1, "Min_FWD"
model += lpSum(x_start[i] for i in indices if df.loc[i, "pos_name"] == "FWD") <= 3, "Max_FWD"

# Max 3 Players per Team
for team in df['team_name'].unique():
    model += lpSum(x_start[i] + x_bench[i] for i in indices if df.loc[i, "team_name"] == team) <= 3, f"Max3_{team}"

# 5. Solve
model.solve()

# 6. Output Formatting
if LpStatus[model.status] == "Optimal":
    starting_xi = []
    bench_squad = []

    for i in indices:
        if x_start[i].varValue == 1:
            row = df.loc[i].to_dict()
            row['Is_Captain'] = True if x_capt[i].varValue == 1 else False
            starting_xi.append(row)
        elif x_bench[i].varValue == 1:
            bench_squad.append(df.loc[i].to_dict())

    start_df = pd.DataFrame(starting_xi)
    bench_df = pd.DataFrame(bench_squad)

    total_cost = (start_df['now_cost'].sum() + bench_df['now_cost'].sum()) / 10.0
    total_pts = start_df['predicted_next_points'].sum() + start_df[start_df['Is_Captain']]['predicted_next_points'].values[0]

    print(f"=== OPTIMAL 15-PLAYER SQUAD SELECTED ===")
    print(f"Total Squad Cost: £{total_cost:.1f}m / £100.0m")
    print(f"Expected Gameweek Score (with Captain): {total_pts:.2f} pts\n")

    print("--- STARTING XI (11 Players) ---")
    print(start_df[['web_name', 'pos_name', 'team_name', 'now_cost', 'predicted_next_points', 'Is_Captain']].to_string(index=False))

    print("\n--- BENCH FODDER (4 Players) ---")
    print(bench_df[['web_name', 'pos_name', 'team_name', 'now_cost', 'predicted_next_points']].to_string(index=False))
else:
    print(f"Solver Status: {LpStatus[model.status]}. Check CSV dataset for position/budget compatibility.")

=== OPTIMAL 15-PLAYER SQUAD SELECTED ===
Total Squad Cost: £82.2m / £100.0m
Expected Gameweek Score (with Captain): 37.88 pts

--- STARTING XI (11 Players) ---
   web_name pos_name      team_name  now_cost  predicted_next_points  Is_Captain
Gibbs-White      MID  Nott'm Forest        79              11.458347        True
      Ajayi      DEF      Hull City        40               2.015765       False
      Mendy      DEF      Hull City        40               2.015765       False
 Matheus N.      DEF       Man City        60               1.710155       False
      Osula      FWD      Newcastle        60               1.612466       False
      Coyle      DEF      Hull City        40               1.601130       False
      Schär      DEF      Newcastle        50               1.523597       False
   Anderson      MID       Man City        64               1.363916       False
  Rodríguez      FWD    Bournemouth        60               1.238477       False
    Wharton      MID Crystal P